In [2]:
!pip install music21

In [3]:
import os
import gc
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import ModelCheckpoint

In [4]:
X = np.load('/kaggle/working/X.npy')

y_onehot = np.load('/kaggle/working/y.npy')
y = np.argmax(y_onehot, axis=1).astype('int32')

del y_onehot
gc.collect()

np.save('/kaggle/working/y_int.npy', y) 

with open('/kaggle/working/note_to_int.json') as f:
    note_to_int = json.load(f)
with open('/kaggle/working/int_to_note.json') as f:
    int_to_note = json.load(f)

vocab_size = len(note_to_int)
print(f"X shape: {X.shape}, y shape: {y.shape}, vocab size: {vocab_size}")

X shape: (728003, 100, 1), y shape: (728003,), vocab size: 2148


In [6]:
strategy = tf.distribute.MirroredStrategy()
print(f"Number of devices: {strategy.num_replicas_in_sync}")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Number of devices: 2


In [7]:
batch_size = 256

dataset = tf.data.Dataset.from_tensor_slices((X, y))
dataset = dataset.shuffle(buffer_size=10000, seed=42)
dataset = dataset.batch(batch_size)
dataset = dataset.prefetch(tf.data.AUTOTUNE)

print(dataset)

<_PrefetchDataset element_spec=(TensorSpec(shape=(None, 100, 1), dtype=tf.float64, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>


In [8]:
checkpoint_path = '/kaggle/working/music_lstm_checkpoint.h5'

checkpoint = ModelCheckpoint(
    checkpoint_path,
    monitor='loss',
    save_best_only=True,
    mode='min'
)

with strategy.scope():
    if os.path.exists(checkpoint_path):
        print("Checkpoint found - resuming from saved weights")
        model = load_model(checkpoint_path)
        model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')
    else:
        print("No checkpoint found - starting fresh")
        model = Sequential()
        model.add(LSTM(256, input_shape=(X.shape[1], X.shape[2]), return_sequences=True))
        model.add(Dropout(0.3))
        model.add(LSTM(256))
        model.add(Dropout(0.3))
        model.add(Dense(256, activation='relu'))
        model.add(Dense(vocab_size, activation='softmax'))
        model.compile(loss='sparse_categorical_crossentropy', optimizer='adam')

Checkpoint found - resuming from saved weights


In [14]:
history = model.fit(
    dataset,
    epochs=24,
    callbacks=[checkpoint]
)

Epoch 1/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.1279

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.1139
Epoch 2/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.1281

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.1120
Epoch 3/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.1263

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 102s 36ms/step - loss: 4.1102
Epoch 4/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 4.1275

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 102s 36ms/step - loss: 4.1084
Epoch 5/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - loss: 4.1202

...

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.0752
Epoch 20/24
2844/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.0904

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.0731
Epoch 21/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.0904

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.0724
Epoch 22/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.0852

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.0691
Epoch 23/24
2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.0697
Epoch 24/24
2843/2844 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - loss: 4.0862

2844/2844 ━━━━━━━━━━━━━━━━━━━━ 103s 36ms/step - loss: 4.0668


In [18]:
import json

model.save('/kaggle/working/music_lstm.keras')
model.save('/kaggle/working/music_lstm.h5')

with open('/kaggle/working/note_mappings.json', 'w') as f:
    json.dump({
        'note_to_int': note_to_int,
        'int_to_note': int_to_note,
        'vocab_size': vocab_size,
        'sequence_length': 100
    }, f)

print("Saved: music_lstm.keras, music_lstm.h5, note_mappings.json")

Saved: music_lstm.keras, music_lstm.h5, note_mappings.json
